In [6]:
# CELL 1 - READ SILVER CLINICAL ENCOUNTERS

silver_clinical_encounters_df = spark.table(
    "silver_clinical_encounters"
)

gold_source_count = silver_clinical_encounters_df.count()

print(
    "Silver Clinical Encounters Rows:",
    gold_source_count
)

silver_clinical_encounters_df.printSchema()

display(
    silver_clinical_encounters_df.limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 8, Finished, Available, Finished, False)

Silver Clinical Encounters Rows: 4862
root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: decimal(18,2) (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: timestamp (nullable = true)
 |-- encounter_type: string (nullable = true)
 |-- facility_country: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- row_hash: string (nullable = true)
 |-- silver_processing_timestamp: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, a6eec549-cea6-4861-8181-eddf4642d4b2)

In [7]:
# CELL 2 - READ SILVER PATIENTS FOR GOLD JOIN

from pyspark.sql.functions import col

silver_patients_df = spark.table(
    "silver_patients"
)

patient_count = silver_patients_df.count()

print("Silver Patients Rows:", patient_count)

# Confirm patient_id is available
if "patient_id" not in silver_patients_df.columns:
    raise ValueError(
        "patient_id is missing from silver_patients."
    )

# Check for duplicate patient IDs before joining
duplicate_patient_ids_df = (
    silver_patients_df
    .groupBy("patient_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_patient_id_count = (
    duplicate_patient_ids_df.count()
)

print(
    "Duplicate patient IDs:",
    duplicate_patient_id_count
)

if duplicate_patient_id_count > 0:
    display(duplicate_patient_ids_df.limit(20))

    raise ValueError(
        "silver_patients contains duplicate patient_id values. "
        "Fix duplicates before performing the Gold join."
    )

print(
    "SUCCESS: Silver patients is ready "
    "to join with clinical encounters."
)

silver_patients_df.printSchema()

display(
    silver_patients_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender",
        "insurance_provider",
        "is_active"
    ).limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 9, Finished, Available, Finished, False)

Silver Patients Rows: 4988
Duplicate patient IDs: 0
SUCCESS: Silver patients is ready to join with clinical encounters.
root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- home_address: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_number: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- last_updated_timestamp: timestamp (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, b1aabc0a-2bae-4eec-a908-375d2637c066)

In [8]:
# CELL 3 - JOIN CLINICAL ENCOUNTERS WITH PATIENT DETAILS
# Left join preserves every clinical encounter

from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp
)

# Select only required patient columns to avoid duplicate metadata columns
patients_for_join_df = (
    silver_patients_df
    .select(
        col("patient_id"),
        col("first_name"),
        col("last_name"),
        col("date_of_birth"),
        col("gender"),
        col("insurance_provider"),
        col("policy_number"),
        col("is_active").alias("patient_is_active")
    )
)

# Join encounters with patients using patient_id
gold_patient_encounters_df = (
    silver_clinical_encounters_df.alias("encounter")
    .join(
        patients_for_join_df.alias("patient"),
        col("encounter.patient_id")
        == col("patient.patient_id"),
        "left"
    )
    .select(
        col("encounter.encounter_id"),
        col("encounter.patient_id"),
        col("patient.first_name"),
        col("patient.last_name"),
        concat_ws(
            " ",
            col("patient.first_name"),
            col("patient.last_name")
        ).alias("patient_full_name"),
        col("patient.date_of_birth"),
        col("patient.gender"),
        col("patient.insurance_provider"),
        col("patient.policy_number"),
        col("patient.patient_is_active"),
        col("encounter.provider_id"),
        col("encounter.facility_id"),
        col("encounter.encounter_timestamp"),
        col("encounter.encounter_type"),
        col("encounter.primary_diagnosis_code"),
        col("encounter.procedure_code"),
        col("encounter.billing_amount"),
        col("encounter.insurance_claim_status"),
        col("encounter.facility_city"),
        col("encounter.facility_country"),
        col("encounter.attending_staff_id"),
        current_timestamp().alias(
            "gold_processing_timestamp"
        )
    )
)

encounter_before_join_count = (
    silver_clinical_encounters_df.count()
)

encounter_after_join_count = (
    gold_patient_encounters_df.count()
)

unmatched_patient_count = (
    gold_patient_encounters_df
    .filter(col("first_name").isNull())
    .count()
)

print(
    "Clinical encounters before join:",
    encounter_before_join_count
)

print(
    "Gold rows after join:",
    encounter_after_join_count
)

print(
    "Encounters without matching patient:",
    unmatched_patient_count
)

# A left join must not change the encounter row count
if encounter_before_join_count != encounter_after_join_count:
    raise ValueError(
        f"Join row-count validation failed. "
        f"Before={encounter_before_join_count}, "
        f"After={encounter_after_join_count}"
    )

print(
    "SUCCESS: Clinical encounters joined with "
    "patient details without changing the row count."
)

display(
    gold_patient_encounters_df.limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 10, Finished, Available, Finished, False)

Clinical encounters before join: 4862
Gold rows after join: 4862
Encounters without matching patient: 953
SUCCESS: Clinical encounters joined with patient details without changing the row count.


SynapseWidget(Synapse.DataFrame, df298952-4a43-424d-ad94-dd19bd0a4508)

In [9]:
# CELL 4 - WRITE JOINED PATIENT ENCOUNTERS GOLD TABLE
# Rerunnable: full refresh replaces the same Gold table

gold_table_name = "gold_patient_encounters"

gold_row_count = gold_patient_encounters_df.count()

print("Gold candidate rows:", gold_row_count)

if gold_row_count == 0:
    raise ValueError(
        "Gold patient encounters contains zero rows."
    )

(
    gold_patient_encounters_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_name)
)

written_row_count = spark.table(
    gold_table_name
).count()

print("Gold table:", gold_table_name)
print("Rows written:", written_row_count)

if gold_row_count != written_row_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate rows={gold_row_count}, "
        f"Written rows={written_row_count}"
    )

print(
    "SUCCESS: gold_patient_encounters was created "
    "without changing the row count."
)

display(
    spark.table(gold_table_name)
    .orderBy("encounter_id")
    .limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 11, Finished, Available, Finished, False)

Gold candidate rows: 4862
Gold table: gold_patient_encounters
Rows written: 4862
SUCCESS: gold_patient_encounters was created without changing the row count.


SynapseWidget(Synapse.DataFrame, 1bacee70-ed13-4649-998a-22a111addd8b)

In [10]:
# CELL 5 - CREATE DAILY ENCOUNTER SUMMARY GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    to_date,
    count,
    countDistinct,
    sum,
    avg,
    current_timestamp,
    col
)

gold_daily_encounter_summary_df = (
    gold_patient_encounters_df
    .withColumn(
        "encounter_date",
        to_date(col("encounter_timestamp"))
    )
    .groupBy(
        "encounter_date",
        "encounter_type"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("patient_id").alias("unique_patients"),
        countDistinct("provider_id").alias("unique_providers"),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount")
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_daily_table_name = "gold_daily_encounter_summary"

candidate_count = (
    gold_daily_encounter_summary_df.count()
)

print("Daily summary candidate rows:", candidate_count)

if candidate_count == 0:
    raise ValueError(
        "Daily encounter summary contains zero rows."
    )

(
    gold_daily_encounter_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_daily_table_name)
)

written_count = spark.table(
    gold_daily_table_name
).count()

print("Gold table:", gold_daily_table_name)
print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Daily encounter summary Gold table created."
)

display(
    spark.table(gold_daily_table_name)
    .orderBy(
        col("encounter_date").desc(),
        col("encounter_type")
    )
    .limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 12, Finished, Available, Finished, False)

Daily summary candidate rows: 78
Gold table: gold_daily_encounter_summary
Rows written: 78
SUCCESS: Daily encounter summary Gold table created.


SynapseWidget(Synapse.DataFrame, 9e171074-1fa2-4542-8f13-fc7733d50a36)

In [11]:
# CELL 6 - CREATE CLAIM STATUS SUMMARY GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    round,
    when,
    col,
    current_timestamp
)

gold_claim_status_summary_df = (
    gold_patient_encounters_df
    .groupBy(
        "insurance_claim_status"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("patient_id").alias("unique_patients"),
        countDistinct("facility_id").alias("unique_facilities"),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount")
    )
    .withColumn(
        "percentage_of_encounters",
        round(
            col("total_encounters")
            / gold_patient_encounters_df.count()
            * 100,
            2
        )
    )
    .withColumn(
        "is_denied_status",
        when(
            col("insurance_claim_status") == "DENIED",
            True
        ).otherwise(False)
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_claim_table_name = "gold_claim_status_summary"

candidate_count = gold_claim_status_summary_df.count()

print("Claim-status summary candidate rows:", candidate_count)

if candidate_count == 0:
    raise ValueError(
        "Claim-status summary contains zero rows."
    )

(
    gold_claim_status_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_claim_table_name)
)

written_count = spark.table(
    gold_claim_table_name
).count()

print("Gold table:", gold_claim_table_name)
print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Claim-status summary Gold table created."
)

display(
    spark.table(gold_claim_table_name)
    .orderBy(
        col("total_encounters").desc()
    )
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 13, Finished, Available, Finished, False)

Claim-status summary candidate rows: 4
Gold table: gold_claim_status_summary
Rows written: 4
SUCCESS: Claim-status summary Gold table created.


SynapseWidget(Synapse.DataFrame, 3b23f8c1-38ac-4013-a890-456b1c03d2f3)

In [12]:
# CELL 7 - CREATE FACILITY PERFORMANCE GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    round,
    when,
    col,
    current_timestamp
)

gold_facility_performance_df = (
    gold_patient_encounters_df
    .groupBy(
        "facility_id",
        "facility_city",
        "facility_country"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("patient_id").alias("unique_patients"),
        countDistinct("provider_id").alias("unique_providers"),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount"),
        sum(
            when(
                col("insurance_claim_status") == "DENIED",
                1
            ).otherwise(0)
        ).alias("denied_claims")
    )
    .withColumn(
        "denial_rate_percentage",
        round(
            col("denied_claims")
            / col("total_encounters")
            * 100,
            2
        )
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_facility_table_name = "gold_facility_performance"

candidate_count = gold_facility_performance_df.count()

print("Facility performance candidate rows:", candidate_count)

if candidate_count == 0:
    raise ValueError(
        "Facility performance summary contains zero rows."
    )

(
    gold_facility_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_facility_table_name)
)

written_count = spark.table(
    gold_facility_table_name
).count()

print("Gold table:", gold_facility_table_name)
print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Facility performance Gold table created."
)

display(
    spark.table(gold_facility_table_name)
    .orderBy(
        col("total_encounters").desc()
    )
    .limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 14, Finished, Available, Finished, False)

Facility performance candidate rows: 2708
Gold table: gold_facility_performance
Rows written: 2708
SUCCESS: Facility performance Gold table created.


SynapseWidget(Synapse.DataFrame, 920e80b5-03d5-49d0-b5af-de30620b8fb4)

In [13]:
# CELL 8 - CREATE DIAGNOSIS SUMMARY GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    round,
    col,
    current_timestamp
)

gold_diagnosis_summary_df = (
    gold_patient_encounters_df
    .groupBy(
        "primary_diagnosis_code"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("patient_id").alias("unique_patients"),
        countDistinct("provider_id").alias("unique_providers"),
        countDistinct("facility_id").alias("unique_facilities"),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount")
    )
    .withColumn(
        "percentage_of_encounters",
        round(
            col("total_encounters")
            / gold_patient_encounters_df.count()
            * 100,
            2
        )
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_diagnosis_table_name = "gold_diagnosis_summary"

candidate_count = gold_diagnosis_summary_df.count()

print("Diagnosis summary candidate rows:", candidate_count)

if candidate_count == 0:
    raise ValueError(
        "Diagnosis summary contains zero rows."
    )

(
    gold_diagnosis_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_diagnosis_table_name)
)

written_count = spark.table(
    gold_diagnosis_table_name
).count()

print("Gold table:", gold_diagnosis_table_name)
print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Diagnosis summary Gold table created."
)

display(
    spark.table(gold_diagnosis_table_name)
    .orderBy(
        col("total_encounters").desc()
    )
    .limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 15, Finished, Available, Finished, False)

Diagnosis summary candidate rows: 20
Gold table: gold_diagnosis_summary
Rows written: 20
SUCCESS: Diagnosis summary Gold table created.


SynapseWidget(Synapse.DataFrame, 65c17b85-da32-4e90-b6f0-3062f35a611c)

In [14]:
# CELL 9 - CREATE ENCOUNTER TYPE SUMMARY GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    round,
    col,
    current_timestamp
)

total_encounter_count = gold_patient_encounters_df.count()

gold_encounter_type_summary_df = (
    gold_patient_encounters_df
    .groupBy(
        "encounter_type"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("patient_id").alias("unique_patients"),
        countDistinct("provider_id").alias("unique_providers"),
        countDistinct("facility_id").alias("unique_facilities"),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount")
    )
    .withColumn(
        "percentage_of_encounters",
        round(
            col("total_encounters")
            / total_encounter_count
            * 100,
            2
        )
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_encounter_type_table_name = (
    "gold_encounter_type_summary"
)

candidate_count = (
    gold_encounter_type_summary_df.count()
)

print(
    "Encounter-type summary candidate rows:",
    candidate_count
)

if candidate_count == 0:
    raise ValueError(
        "Encounter-type summary contains zero rows."
    )

(
    gold_encounter_type_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_encounter_type_table_name)
)

written_count = spark.table(
    gold_encounter_type_table_name
).count()

print(
    "Gold table:",
    gold_encounter_type_table_name
)

print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Encounter-type summary "
    "Gold table created."
)

display(
    spark.table(
        gold_encounter_type_table_name
    )
    .orderBy(
        col("total_encounters").desc()
    )
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 16, Finished, Available, Finished, False)

Encounter-type summary candidate rows: 3
Gold table: gold_encounter_type_summary
Rows written: 3
SUCCESS: Encounter-type summary Gold table created.


SynapseWidget(Synapse.DataFrame, f23eb82a-577d-4b6d-b178-3b5672432e0a)

In [15]:
# CELL 10 - CREATE PATIENT ENCOUNTER SUMMARY GOLD TABLE
# Rerunnable: full refresh

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    max,
    min,
    col,
    current_timestamp
)

gold_patient_summary_df = (
    gold_patient_encounters_df
    .groupBy(
        "patient_id",
        "patient_full_name",
        "date_of_birth",
        "gender",
        "insurance_provider",
        "patient_is_active"
    )
    .agg(
        count("*").alias("total_encounters"),
        countDistinct("provider_id").alias("unique_providers"),
        countDistinct("facility_id").alias("unique_facilities"),
        countDistinct("primary_diagnosis_code").alias(
            "unique_diagnoses"
        ),
        sum("billing_amount").alias("total_billing_amount"),
        avg("billing_amount").alias("average_billing_amount"),
        min("encounter_timestamp").alias("first_encounter_timestamp"),
        max("encounter_timestamp").alias("latest_encounter_timestamp")
    )
    .withColumn(
        "gold_processing_timestamp",
        current_timestamp()
    )
)

gold_patient_summary_table_name = (
    "gold_patient_encounter_summary"
)

candidate_count = gold_patient_summary_df.count()

print("Patient summary candidate rows:", candidate_count)

if candidate_count == 0:
    raise ValueError(
        "Patient encounter summary contains zero rows."
    )

(
    gold_patient_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_patient_summary_table_name)
)

written_count = spark.table(
    gold_patient_summary_table_name
).count()

print("Gold table:", gold_patient_summary_table_name)
print("Rows written:", written_count)

if candidate_count != written_count:
    raise ValueError(
        f"Gold write validation failed. "
        f"Candidate={candidate_count}, "
        f"Written={written_count}"
    )

print(
    "SUCCESS: Patient encounter summary "
    "Gold table created."
)

display(
    spark.table(gold_patient_summary_table_name)
    .orderBy(
        col("total_encounters").desc()
    )
    .limit(20)
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 17, Finished, Available, Finished, False)

Patient summary candidate rows: 3129
Gold table: gold_patient_encounter_summary
Rows written: 3129
SUCCESS: Patient encounter summary Gold table created.


SynapseWidget(Synapse.DataFrame, 9e6dc12c-79e8-4573-a8a9-50db34a0908e)

In [16]:
# CELL 11 - FINAL GOLD CLINICAL ENCOUNTERS VALIDATION
# Read-only checks for all Gold tables

from pyspark.sql.functions import col

gold_tables = {
    "gold_patient_encounters": "encounter_id",
    "gold_daily_encounter_summary": "encounter_date",
    "gold_claim_status_summary": "insurance_claim_status",
    "gold_facility_performance": "facility_id",
    "gold_diagnosis_summary": "primary_diagnosis_code",
    "gold_encounter_type_summary": "encounter_type",
    "gold_patient_encounter_summary": "patient_id"
}

missing_tables = [
    table_name
    for table_name in gold_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_tables:
    raise ValueError(
        f"Missing Gold tables: {missing_tables}"
    )

silver_encounter_count = (
    silver_clinical_encounters_df.count()
)

gold_detail_count = spark.table(
    "gold_patient_encounters"
).count()

print("Silver encounter rows:", silver_encounter_count)
print("Gold detail rows:", gold_detail_count)

if silver_encounter_count != gold_detail_count:
    raise ValueError(
        f"Gold detail reconciliation failed. "
        f"Silver={silver_encounter_count}, "
        f"Gold={gold_detail_count}"
    )

# Check duplicate encounter IDs in the detailed Gold table
duplicate_gold_encounters = (
    spark.table("gold_patient_encounters")
    .groupBy("encounter_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Duplicate Gold encounter IDs:",
    duplicate_gold_encounters
)

if duplicate_gold_encounters > 0:
    raise ValueError(
        f"Found {duplicate_gold_encounters} duplicate "
        "encounter IDs in gold_patient_encounters."
    )

print("=" * 60)
print("GOLD CLINICAL ENCOUNTERS TABLE SUMMARY")
print("=" * 60)

for table_name in gold_tables:
    table_count = spark.table(table_name).count()

    print(
        f"{table_name}: {table_count} rows"
    )

    if table_count == 0:
        raise ValueError(
            f"{table_name} contains zero rows."
        )

print("=" * 60)

print(
    "SUCCESS: All clinical encounter Gold tables "
    "exist, contain data and passed reconciliation."
)

StatementMeta(, def50fd3-6120-4b2d-a535-180d57bf3291, 18, Finished, Available, Finished, False)

Silver encounter rows: 4862
Gold detail rows: 4862
Duplicate Gold encounter IDs: 0
GOLD CLINICAL ENCOUNTERS TABLE SUMMARY
gold_patient_encounters: 4862 rows
gold_daily_encounter_summary: 78 rows
gold_claim_status_summary: 4 rows
gold_facility_performance: 2708 rows
gold_diagnosis_summary: 20 rows
gold_encounter_type_summary: 3 rows
gold_patient_encounter_summary: 3129 rows
SUCCESS: All clinical encounter Gold tables exist, contain data and passed reconciliation.
